# Block PII column access for the analytics role using Lake Formation column-level grants

Compliance flagged the customers table on Friday: the analytics role can read `ssn` and `email` because the underlying S3 bucket policy is too broad. Lake Formation column grants close this without breaking the rest of the queries. You have one session to migrate the table to LF governance, scope the analytics grants, and prove via Athena that PII reads are denied.

The four deliverables map to four checkpoints:
1. The lab's S3 location is registered with Lake Formation, and the `IAMAllowedPrincipals` default grant is removed from the customers table so LF grants are no longer shadowed by the legacy IAM-allowed path.
2. The Glue table `customers` is registered over the seeded Parquet with columns `customer_id`, `ssn`, `email`, `signup_date`, and `country`.
3. The analytics role holds a Lake Formation `SELECT` grant on the customers table with `ssn` and `email` excluded from the column wildcard (or, equivalently, an explicit `ColumnNames` list of the three non-PII columns).
4. The analytics role, assumed via STS and used to build a fresh Athena client, can run `SELECT customer_id, country FROM customers LIMIT 10` and is blocked from running `SELECT ssn FROM customers LIMIT 10` with an `AccessDenied` or `Insufficient permissions` error.

**Time.** About 55 minutes hands-on. Lake Formation grants are control-plane calls and return in seconds; the only wall-clock cost is the two Athena queries in Task 4, which each take a few seconds to plan and execute.

**Cost.** This lab costs effectively nothing. Lake Formation does not charge for grants or registrations. You pay for the underlying S3 (kilobytes), Glue Data Catalog calls (free at this scale), and three or four small Athena queries (fractions of a penny). A realistic session lands under 5 cents.

This lab maps to AWS DEA-C01 Domain 4: Data Security and Governance (18% of exam weight).


In [ ]:
# NBVAL_SKIP
# Install labrun-checks. Pinned to a specific version per LAB-CREATION-BLUEPRINT.md
# build rules. Never use unpinned installs.

!pip install --quiet labrun-checks==0.7.0


In [ ]:
# NBVAL_SKIP
# Imports and per-lab constants. Resource names follow the
# labrun-{lab-slug}-{descriptor} pattern from LAB-CREATION-BLUEPRINT.md
# section 12.

import atexit
import getpass
import io
import json
import time

import boto3
from botocore.exceptions import ClientError

from labrun_checks import (
    register,
    check,
    cleanup,
    run_cleanup,
    CleanupResource,
    CheckpointResult,
)

LAB_ID = "lake-formation-column-permissions"
LAB_TAG_KEY = "labrun:lab-slug"
LAB_TAG_VALUE = LAB_ID  # must equal cert YAML labs[10].slug exactly
REGION = "us-east-1"  # all DEA-C01 labs run in us-east-1 per LAB-CREATION-BLUEPRINT section 15

# Resource names. Bucket name appends the account ID for global uniqueness.
DATABASE_NAME = f"labrun_{LAB_ID.replace('-', '_')}_db"
TABLE_NAME = "customers"
ANALYTICS_ROLE_NAME = f"labrun-{LAB_ID}-analytics-role"
WORKGROUP_NAME = f"labrun-{LAB_ID}-wg"
BUCKET_NAME = None  # resolved after STS once the account ID is known

# PII columns the analytics role must not be able to read.
PII_COLUMNS = ["ssn", "email"]
NON_PII_COLUMNS = ["customer_id", "signup_date", "country"]

# Seed config. 100 deterministic rows are plenty to demonstrate the column-level
# grant boundary without paying for a real dataset.
SEED = 42
ROWS = 100

# Lake Formation grant bodies stored here so the inline cleanup adapter can
# look them up by CleanupResource.id and issue revoke_permissions with the
# exact structured form. LF requires the precise grant body to revoke; a
# stringified CLI command in cli_delete_command is for documentation only.
LF_GRANT_BODIES: dict = {}


In [ ]:
# NBVAL_SKIP
# Register the session, validate AWS credentials via STS GetCallerIdentity,
# confirm the region, and seed Lake Formation as a data lake admin. This cell
# must succeed before the manifest cell creates anything per
# LAB-CREATION-BLUEPRINT section 15.

session_token = getpass.getpass("Paste your session token from labrun.dev: ")
aws_access_key_id = getpass.getpass("AWS_ACCESS_KEY_ID: ")
aws_secret_access_key = getpass.getpass("AWS_SECRET_ACCESS_KEY: ")
aws_session_token = getpass.getpass(
    "AWS_SESSION_TOKEN (leave blank for long-lived credentials): "
).strip()
region_input = input(f"AWS region (default {REGION}): ").strip()
if region_input and region_input != REGION:
    print(f"Region {region_input!r} is not supported for this lab.")
    print(f"All DEA-C01 labs run in {REGION} (N. Virginia).")
    print("Re-run this cell and accept the default.")
    raise SystemExit(1)

AWS_CREDENTIALS = {
    "aws_access_key_id": aws_access_key_id,
    "aws_secret_access_key": aws_secret_access_key,
    "region": REGION,
}
if aws_session_token:
    AWS_CREDENTIALS["aws_session_token"] = aws_session_token

# Validate credentials against AWS via STS GetCallerIdentity. Fail fast with a
# clear message rather than waiting for the first IAM call to error out.
sts = boto3.client(
    "sts",
    aws_access_key_id=aws_access_key_id,
    aws_secret_access_key=aws_secret_access_key,
    aws_session_token=aws_session_token or None,
    region_name=REGION,
)
try:
    identity = sts.get_caller_identity()
except ClientError as e:
    print("Credentials are missing or expired. STS GetCallerIdentity failed:")
    print(f"  {e}")
    print()
    print("Refresh your AWS credentials and re-run this cell.")
    raise SystemExit(1)

ACCOUNT_ID = identity["Account"]
CALLER_ARN = identity["Arn"]
print(f"Credentials validated. Account: {ACCOUNT_ID}")
print(f"Caller ARN: {CALLER_ARN}")
print(f"Region: {REGION}")
print(f"Session token in use: {bool(aws_session_token)}")

# Resolve the bucket name now that we know the account ID.
BUCKET_NAME = f"labrun-{LAB_ID}-{ACCOUNT_ID}"
print(f"Bucket name resolved: {BUCKET_NAME}")

# Lake Formation first-time setup. The first time LF is touched in an account,
# AWS requires a data lake administrator before any grant call will succeed.
# Without this, every subsequent grant returns AccessDeniedException and the
# failure looks like an IAM problem. put_data_lake_settings is idempotent.
lakeformation_setup = boto3.client(
    "lakeformation",
    aws_access_key_id=aws_access_key_id,
    aws_secret_access_key=aws_secret_access_key,
    aws_session_token=aws_session_token or None,
    region_name=REGION,
)
try:
    current = lakeformation_setup.get_data_lake_settings()
    admins = current.get("DataLakeSettings", {}).get("DataLakeAdmins", [])
    admin_ids = {a.get("DataLakePrincipalIdentifier") for a in admins}
    if CALLER_ARN not in admin_ids:
        merged = list(admins) + [{"DataLakePrincipalIdentifier": CALLER_ARN}]
        lakeformation_setup.put_data_lake_settings(
            DataLakeSettings={"DataLakeAdmins": merged},
        )
        print(f"Added {CALLER_ARN} as a Lake Formation data lake admin.")
    else:
        print(f"Caller is already a Lake Formation data lake admin.")
except ClientError as e:
    code = e.response["Error"]["Code"]
    if code in ("AccessDeniedException", "EntityNotFoundException"):
        # First-time setup in an account with no settings yet. Set the caller
        # as the sole admin so subsequent grant calls do not 403.
        lakeformation_setup.put_data_lake_settings(
            DataLakeSettings={
                "DataLakeAdmins": [{"DataLakePrincipalIdentifier": CALLER_ARN}],
            },
        )
        print(f"Initialized Lake Formation with {CALLER_ARN} as data lake admin.")
    else:
        raise

# Register the session with labrun-checks. register() returns None;
# do not assign its return value.
register(session_token=session_token, lab_id=LAB_ID, credentials=AWS_CREDENTIALS)
print(f"Session registered for lab_id: {LAB_ID}")


In [ ]:
# NBVAL_SKIP
# Cleanup manifest, atexit safety net, and orphan scan.
# The manifest is module-level and in strict reverse-creation order.
# Lab 11 has no critical (hourly-billed) resources. Lake Formation grants
# and registrations accrue no per-hour or per-month charge, but they are
# account-level state and MUST be explicitly cleaned up.
#
# Per RESOURCE-SAFETY-SPEC Rule 4, the orphan scan blocks execution if any
# tagged resources from a prior session exist (not just print a warning).
#
# Note: labrun-checks 0.3.0 ships AWS adapters for s3_bucket, iam_role,
# glue_database, and glue_table. It does NOT yet support
# lakeformation_permissions, lakeformation_resource, or athena_workgroup.
# A _LabAdapter subclass extending AwsCleanupAdapter is defined in the
# cleanup cell at the bottom of this notebook and passed to run_cleanup.
# The new resource types are still declared declaratively here so the
# canonical summary, atexit handler, and orphan scan all see them.
#
# The lakeformation_permissions adapter requires the structured grant body
# (Principal, Resource.TableWithColumns, Permissions) to call
# revoke_permissions. Task 3 stores that body in LF_GRANT_BODIES keyed by
# CleanupResource.id so the adapter can look it up at teardown.

CLEANUP_MANIFEST = [
    CleanupResource(
        type="lakeformation_permissions",
        id=f"analytics-role:{DATABASE_NAME}.{TABLE_NAME}",
        region=REGION,
        cli_delete_command=(
            "aws lakeformation revoke-permissions "
            "--principal DataLakePrincipalIdentifier=<analytics-role-arn> "
            f"--resource '{{\"TableWithColumns\": {{...}}}}' "
            "--permissions SELECT (full JSON in adapter)"
        ),
    ),
    CleanupResource(
        type="lakeformation_resource",
        id=f"arn:aws:s3:::{BUCKET_NAME}/",
        region=REGION,
        cli_delete_command=(
            f"aws lakeformation deregister-resource "
            f"--resource-arn arn:aws:s3:::{BUCKET_NAME}/"
        ),
    ),
    CleanupResource(
        type="athena_workgroup",
        id=WORKGROUP_NAME,
        region=REGION,
        cli_delete_command=(
            f"aws athena delete-work-group --work-group {WORKGROUP_NAME} "
            f"--recursive-delete-option"
        ),
    ),
    CleanupResource(
        type="glue_table",
        id=TABLE_NAME,
        region=REGION,
        parent=DATABASE_NAME,
        cli_delete_command=(
            f"aws glue delete-table --database-name {DATABASE_NAME} "
            f"--name {TABLE_NAME}"
        ),
    ),
    CleanupResource(
        type="glue_database",
        id=DATABASE_NAME,
        region=REGION,
        cli_delete_command=f"aws glue delete-database --name {DATABASE_NAME}",
    ),
    CleanupResource(
        type="iam_role",
        id=ANALYTICS_ROLE_NAME,
        region=REGION,
        cli_delete_command=f"aws iam delete-role --role-name {ANALYTICS_ROLE_NAME}",
    ),
    CleanupResource(
        type="s3_bucket",
        id=BUCKET_NAME,
        region=REGION,
        cli_delete_command=f"aws s3 rb s3://{BUCKET_NAME} --force",
    ),
]


def _atexit_cleanup() -> None:
    """Best-effort teardown on kernel shutdown.

    The dedicated cleanup cell is the authoritative entry point; this is the
    safety net for kernel crashes. Errors are swallowed because atexit
    handlers must not raise.
    """
    try:
        run_cleanup(CLEANUP_MANIFEST, adapter=_LabAdapter()) if "_LabAdapter" in globals() else run_cleanup(CLEANUP_MANIFEST)
    except Exception:
        pass


atexit.register(_atexit_cleanup)


def scan_for_orphans() -> list[str]:
    """Refuse to start if a previous run left tagged resources behind.

    Per RESOURCE-SAFETY-SPEC Rule 4, the setup cell must check for leftover
    state with the lab's tag and surface the cleanup command before creating
    any new resources.
    """
    client = boto3.client(
        "resourcegroupstaggingapi",
        aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
        aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
        aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
        region_name=REGION,
    )
    arns: list[str] = []
    paginator = client.get_paginator("get_resources")
    for page in paginator.paginate(
        TagFilters=[{"Key": LAB_TAG_KEY, "Values": [LAB_TAG_VALUE]}],
    ):
        for item in page.get("ResourceTagMappingList", []):
            arns.append(item.get("ResourceARN", ""))
    return arns


orphans = scan_for_orphans()
if orphans:
    print(f"BLOCKED: existing resources tagged labrun:lab-slug={LAB_TAG_VALUE} were found:")
    for arn in orphans:
        print("  -", arn)
    print()
    print("These are leftovers from a previous run of this lab. Re-running")
    print("setup against an unclean state can produce duplicate Lake Formation")
    print("grants or collide on the role and workgroup names. Run the cleanup")
    print("cell at the bottom of this notebook first, or remove them manually")
    print("with the AWS CLI commands the cleanup cell prints, then re-run setup.")
    raise SystemExit(1)

print("No prior orphans found. Safe to create resources for this session.")


## Task 1: Register the S3 location with Lake Formation and revoke the `IAMAllowedPrincipals` default grant

Lake Formation can only enforce column-level grants on tables whose underlying S3 location is **registered** with LF and whose default `IAMAllowedPrincipals` grant has been **revoked**. Until both happen, the catalog runs in legacy IAM mode and column grants are silently shadowed. This task lands both flips and is the single biggest source of confusion in LF labs.

Three pieces of plumbing to land in this task:

1. **Create the lab S3 bucket**, tag it, and seed `customers/customers.parquet` with the 100-row dataset. The Parquet is built in memory with pyarrow; the column order in the file becomes the column order the Glue table sees.
2. **Create the LF service-linked role**, if it does not already exist, so Lake Formation can read from S3 on the analytics role's behalf at query time. The bundled `register_resource` call carries `UseServiceLinkedRole=True` so AWS provisions the role lazily on first use.
3. **Register the S3 location with Lake Formation** by calling `lakeformation.register_resource(ResourceArn=arn:aws:s3:::{BUCKET}/, UseServiceLinkedRole=True)`. Then **revoke the `IAMAllowedPrincipals` default permissions** from the database that will hold the customers table. The default-permissions revoke is on the catalog scope; the table revoke happens in Task 2 right after the table is created. Until this revoke fires, LF grants are ignored.

After this task, the bucket is the LF-governed source for the table, and any IAM principal that previously inherited `IAMAllowedPrincipals` access loses that path.


In [ ]:
# NBVAL_SKIP
# Task 1: Create the bucket, seed the customers Parquet, create the Glue
# database, register the S3 location with Lake Formation, and revoke the
# IAMAllowedPrincipals default grant from the database.

s3 = boto3.client(
    "s3",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)
glue = boto3.client(
    "glue",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)
lakeformation = boto3.client(
    "lakeformation",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)

# Create the bucket. us-east-1 rejects LocationConstraint; other regions
# require it. All DEA-C01 labs run in us-east-1, so the simple call works.
# YOUR CODE: Create the S3 bucket with s3.create_bucket(Bucket=BUCKET_NAME).

s3.put_bucket_tagging(
    Bucket=BUCKET_NAME,
    Tagging={"TagSet": [{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}]},
)
print(f"Bucket created and tagged: {BUCKET_NAME}")

# Lazy pyarrow import; Colab ships it preinstalled. Local kernels may need
# pip install pyarrow.
try:
    import pyarrow as pa
    import pyarrow.parquet as pq
except ImportError:
    print("pyarrow is not installed. Run !pip install pyarrow and re-run this cell.")
    raise

# Deterministic synthetic data. Column order in the Parquet becomes the
# column order in the Glue table StorageDescriptor.
import random as _random
_random.seed(SEED)

COUNTRIES = ["US", "CA", "UK", "DE", "FR", "JP", "AU", "BR"]
customer_ids = [f"cust_{i:04d}" for i in range(ROWS)]
ssns = [f"{_random.randint(100, 999)}-{_random.randint(10, 99)}-{_random.randint(1000, 9999)}" for _ in range(ROWS)]
emails = [f"user{i}@example.com" for i in range(ROWS)]
signup_dates = []
BASE_EPOCH = 1730419200  # 2024-11-01T00:00:00Z
for i in range(ROWS):
    epoch = BASE_EPOCH + i * 86400
    signup_dates.append(time.strftime("%Y-%m-%d", time.gmtime(epoch)))
countries = [_random.choice(COUNTRIES) for _ in range(ROWS)]

customers_table = pa.table({
    "customer_id": customer_ids,
    "ssn": ssns,
    "email": emails,
    "signup_date": signup_dates,
    "country": countries,
})

buf = io.BytesIO()
pq.write_table(customers_table, buf, compression="snappy")
buf.seek(0)
parquet_bytes = buf.getvalue()

CUSTOMERS_KEY = "customers/customers.parquet"
s3.put_object(Bucket=BUCKET_NAME, Key=CUSTOMERS_KEY, Body=parquet_bytes)
print(f"Seeded {ROWS} rows at s3://{BUCKET_NAME}/{CUSTOMERS_KEY} ({len(parquet_bytes)} bytes)")

# Glue database for the customers table. Glue Data Catalog is free; creating
# the database is a control-plane call.
try:
    glue.create_database(
        DatabaseInput={
            "Name": DATABASE_NAME,
            "Description": f"Catalog for {LAB_ID}",
        },
    )
    glue.tag_resource(
        ResourceArn=f"arn:aws:glue:{REGION}:{ACCOUNT_ID}:database/{DATABASE_NAME}",
        TagsToAdd={LAB_TAG_KEY: LAB_TAG_VALUE},
    )
    print(f"Glue database created and tagged: {DATABASE_NAME}")
except ClientError as e:
    if e.response["Error"]["Code"] == "AlreadyExistsException":
        print(f"Glue database {DATABASE_NAME} already exists, reusing.")
    else:
        raise

# Register the S3 location with Lake Formation. UseServiceLinkedRole=True
# tells AWS to lazily provision AWSServiceRoleForLakeFormationDataAccess so
# LF can read from S3 on the analytics role's behalf at query time.
bucket_path_arn = f"arn:aws:s3:::{BUCKET_NAME}/"

# YOUR CODE: Register the S3 location with
# lakeformation.register_resource(
#     ResourceArn=bucket_path_arn,
#     UseServiceLinkedRole=True,
# ).
# Wrap in try/except ClientError; on AlreadyExistsException the location is
# already registered from a prior run inside this session and that is fine.

print(f"S3 location registered with Lake Formation: {bucket_path_arn}")

# Revoke the IAMAllowedPrincipals default grant from the database. Until this
# revoke fires, every new table inside the database inherits the legacy
# IAM-allowed path and LF column grants are silently ignored. Same revoke
# pattern fires against the table in Task 2 right after the table is created.
try:
    lakeformation.revoke_permissions(
        Principal={"DataLakePrincipalIdentifier": "IAM_ALLOWED_PRINCIPALS"},
        Resource={"Database": {"Name": DATABASE_NAME}},
        Permissions=["ALL"],
        PermissionsWithGrantOption=[],
    )
    print(f"Revoked IAMAllowedPrincipals default grant from database {DATABASE_NAME}.")
except ClientError as e:
    code = e.response["Error"]["Code"]
    if code in ("InvalidInputException",):
        # The grant did not exist on the database. Treat as already-revoked.
        print(f"No IAMAllowedPrincipals grant present on database {DATABASE_NAME}; nothing to revoke.")
    else:
        raise


In [ ]:
# NBVAL_SKIP
# Checkpoint 1: S3 location is registered with Lake Formation and the
# IAMAllowedPrincipals default grant has been removed from the customers
# table's default permissions. The table itself may not exist yet at this
# point (it is created in Task 2); the validator accepts both cases and
# only walks the table-level grants if the table is present.


def checkpoint_1(session):
    try:
        lf = boto3.client(
            "lakeformation",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )

        bucket_path_arn = f"arn:aws:s3:::{BUCKET_NAME}/"

        # Step 1: list_resources shows the lab bucket registered.
        registered = False
        paginator = lf.get_paginator("list_resources")
        for page in paginator.paginate():
            for r in page.get("ResourceInfoList", []):
                if r.get("ResourceArn", "").rstrip("/") == bucket_path_arn.rstrip("/"):
                    registered = True
                    break
            if registered:
                break

        if not registered:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Lake Formation list_resources does not include "
                    f"{bucket_path_arn}. Run lakeformation.register_resource("
                    f"ResourceArn={bucket_path_arn!r}, UseServiceLinkedRole=True)."
                ),
            )

        # Step 2: confirm IAMAllowedPrincipals is NOT a grant on the customers
        # table. If the table does not exist yet, fall back to checking the
        # database's default permissions.
        glue_client = boto3.client(
                "glue",
                aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
                aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
                aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
                region_name=REGION,
        )
        table_exists = False
        try:
            glue_client.get_table(DatabaseName=DATABASE_NAME, Name=TABLE_NAME)
            table_exists = True
        except ClientError as e:
            if e.response["Error"]["Code"] != "EntityNotFoundException":
                return CheckpointResult(status="error", error_reason=str(e))

        # Walk grants on whichever scope is available.
        if table_exists:
            resource = {"Table": {"DatabaseName": DATABASE_NAME, "Name": TABLE_NAME}}
            scope_label = f"table {DATABASE_NAME}.{TABLE_NAME}"
        else:
            resource = {"Database": {"Name": DATABASE_NAME}}
            scope_label = f"database {DATABASE_NAME}"

        try:
            perms = lf.list_permissions(Resource=resource)
        except ClientError as e:
            return CheckpointResult(
                status="error",
                error_reason=f"list_permissions failed for {scope_label}: {e}",
            )

        for grant in perms.get("PrincipalResourcePermissions", []):
            principal = grant.get("Principal", {}).get("DataLakePrincipalIdentifier", "")
            if principal == "IAM_ALLOWED_PRINCIPALS":
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"IAMAllowedPrincipals still has a grant on {scope_label}. "
                        f"Run lakeformation.revoke_permissions("
                        f"Principal={{'DataLakePrincipalIdentifier': 'IAM_ALLOWED_PRINCIPALS'}}, "
                        f"Resource={resource!r}, Permissions=['ALL'])."
                    ),
                )

        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(1, checkpoint_1)


<details><summary>Hint 1 (nudge)</summary>

The checkpoint runs two independent checks: the S3 path ARN appears in `lakeformation.list_resources`, and `IAMAllowedPrincipals` does not appear as a `DataLakePrincipalIdentifier` on the customers table (or, if the table is not built yet, on the database). Read the failure reason. The most common miss on the first run is the bucket not being created, which surfaces downstream as `register_resource` failing to find the ARN.

</details>

<details><summary>Hint 2 (direction)</summary>

Two API calls hold the win in this task. `s3.create_bucket(Bucket=BUCKET_NAME)` builds the bucket; the rest of the cell does the put_object and the LF setup. `lakeformation.register_resource(ResourceArn=bucket_path_arn, UseServiceLinkedRole=True)` registers the path with LF. Wrap that call in try/except ClientError and on `AlreadyExistsException` swallow the error so a re-run does not abort. The IAMAllowedPrincipals revoke on the database is already in the cell as a structured call.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Here is the complete working solution for Task 1 (the seed, the Glue database create, and the database-scope revoke are already in the cell):

```python
s3.create_bucket(Bucket=BUCKET_NAME)
s3.put_bucket_tagging(
    Bucket=BUCKET_NAME,
    Tagging={"TagSet": [{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}]},
)

s3.put_object(Bucket=BUCKET_NAME, Key=CUSTOMERS_KEY, Body=parquet_bytes)

try:
    lakeformation.register_resource(
        ResourceArn=bucket_path_arn,
        UseServiceLinkedRole=True,
    )
except ClientError as e:
    if e.response["Error"]["Code"] != "AlreadyExistsException":
        raise
```

If `register_resource` returns `AccessDeniedException`, your caller is not yet a Lake Formation data lake admin. Re-run the setup cell; `put_data_lake_settings` is the call that grants admin status, and it is idempotent. If `revoke_permissions` returns `InvalidInputException` with "no permissions to revoke," the database was created after the IAMAllowedPrincipals default was already disabled at the account level; treat that case as a pass-through.

</details>


**Wallet check.** Still at $0.00. Lake Formation does not charge for grants or registrations. The S3 bucket is free; the seeded 100-row Parquet is well under a kilobyte. The Glue database is a control-plane object and Glue Data Catalog is free at this scale. Coffee continues to win.


## Task 2: Register the `customers` table in the Glue Data Catalog

The Parquet file is already in S3 from Task 1. Now wire it up as a Glue table so Athena and Lake Formation can both reach it.

Three pieces of plumbing to land in this task:

1. **Build the table input** with explicit `Columns` for `customer_id`, `ssn`, `email`, `signup_date`, `country` (in that order), the Parquet input/output formats, and the `Location` pointing at `s3://{BUCKET}/customers/`. Lake Formation column grants key on column **names**, so the spelling here must match the Parquet schema exactly.
2. **Create the table** with `glue.create_table(DatabaseName=DATABASE_NAME, TableInput=table_input)`. Wrap in try/except on `AlreadyExistsException` so a re-run does not abort.
3. **Revoke `IAMAllowedPrincipals` from the new table** so its column grants are not silently shadowed. AWS auto-adds the default grant on every new table; the revoke must fire right after the create.

After this task the catalog has the table, the table is under LF governance, and the path is set for Task 3 to grant the analytics role column-level SELECT.


In [ ]:
# NBVAL_SKIP
# Task 2: Create the customers Glue table over the seeded Parquet, then
# revoke IAMAllowedPrincipals from the new table so LF column grants are
# not shadowed.

glue = boto3.client(
    "glue",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)
lakeformation = boto3.client(
    "lakeformation",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)

# Table input. Column order matches the Parquet column order from Task 1.
# Lake Formation column grants key on column names, so the spelling must
# match the Parquet schema exactly.
table_input = {
    "Name": TABLE_NAME,
    "TableType": "EXTERNAL_TABLE",
    "Parameters": {
        "classification": "parquet",
        "has_encrypted_data": "false",
    },
    "StorageDescriptor": {
        "Columns": [
            {"Name": "customer_id", "Type": "string"},
            {"Name": "ssn", "Type": "string"},
            {"Name": "email", "Type": "string"},
            {"Name": "signup_date", "Type": "string"},
            {"Name": "country", "Type": "string"},
        ],
        "Location": f"s3://{BUCKET_NAME}/customers/",
        "InputFormat": "org.apache.hadoop.hive.ql.io.parquet.MapredParquetInputFormat",
        "OutputFormat": "org.apache.hadoop.hive.ql.io.parquet.MapredParquetOutputFormat",
        "SerdeInfo": {
            "SerializationLibrary": "org.apache.hadoop.hive.ql.io.parquet.serde.ParquetHiveSerDe",
            "Parameters": {"serialization.format": "1"},
        },
    },
    "PartitionKeys": [],
}

# YOUR CODE: Register the table with glue.create_table(
#     DatabaseName=DATABASE_NAME,
#     TableInput=table_input,
# ).
# Wrap in try/except ClientError; on AlreadyExistsException reuse.

print(f"Glue table created: {DATABASE_NAME}.{TABLE_NAME}")

# Revoke IAMAllowedPrincipals from the new table. AWS auto-adds the default
# grant on every new table; without this revoke the LF column grant from
# Task 3 would be silently ignored and Checkpoint 4 would surface an
# AccessDenied-shaped failure as a permissions success.
try:
    lakeformation.revoke_permissions(
        Principal={"DataLakePrincipalIdentifier": "IAM_ALLOWED_PRINCIPALS"},
        Resource={
            "Table": {"DatabaseName": DATABASE_NAME, "Name": TABLE_NAME},
        },
        Permissions=["ALL"],
        PermissionsWithGrantOption=[],
    )
    print(f"Revoked IAMAllowedPrincipals from table {DATABASE_NAME}.{TABLE_NAME}.")
except ClientError as e:
    code = e.response["Error"]["Code"]
    if code in ("InvalidInputException",):
        # No grant present. Treat as already-revoked.
        print(f"No IAMAllowedPrincipals grant present on table {DATABASE_NAME}.{TABLE_NAME}; nothing to revoke.")
    else:
        raise


In [ ]:
# NBVAL_SKIP
# Checkpoint 2: Glue table customers exists with the five required columns
# and the StorageDescriptor.Location points at s3://{BUCKET}/customers/.

REQUIRED_COLUMNS = {"customer_id", "ssn", "email", "signup_date", "country"}


def checkpoint_2(session):
    try:
        glue_client = boto3.client(
            "glue",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )

        try:
            resp = glue_client.get_table(DatabaseName=DATABASE_NAME, Name=TABLE_NAME)
        except ClientError as e:
            code = e.response["Error"]["Code"]
            if code == "EntityNotFoundException":
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"Glue table {DATABASE_NAME}.{TABLE_NAME} does not exist. "
                        f"Run the Task 2 cell."
                    ),
                )
            return CheckpointResult(status="error", error_reason=str(e))

        table = resp["Table"]
        sd = table.get("StorageDescriptor", {})
        cols = {c.get("Name", "") for c in sd.get("Columns", [])}

        missing = REQUIRED_COLUMNS - cols
        if missing:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Glue table {TABLE_NAME} is missing columns: {sorted(missing)}. "
                    f"Found columns: {sorted(cols)}. Re-build the table input with the "
                    f"five required columns in the StorageDescriptor."
                ),
            )

        expected_location = f"s3://{BUCKET_NAME}/customers/"
        actual_location = sd.get("Location", "")
        # Accept with or without trailing slash, both are valid Glue patterns.
        if actual_location.rstrip("/") != expected_location.rstrip("/"):
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"StorageDescriptor.Location is {actual_location!r}, "
                    f"expected {expected_location!r}."
                ),
            )

        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(2, checkpoint_2)


<details><summary>Hint 1 (nudge)</summary>

The checkpoint walks the table definition in order: table exists, the five required columns are present (`customer_id`, `ssn`, `email`, `signup_date`, `country`), and `StorageDescriptor.Location` matches `s3://{BUCKET}/customers/`. Read the failure reason. The most common miss is a typo in a column name; LF column grants in Task 3 key on the spelling, so a mismatch here surfaces later as a confusing "column not found" error.

</details>

<details><summary>Hint 2 (direction)</summary>

One API call carries the task. `glue.create_table(DatabaseName=DATABASE_NAME, TableInput=table_input)` where `table_input` is the dict already defined in the cell. Wrap in try/except `ClientError` and on `AlreadyExistsException` reuse the table rather than re-creating. The Parquet serde and input/output format strings are already in the dict; do not rewrite them. The revoke of `IAMAllowedPrincipals` on the new table is already in the cell.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Here is the complete working solution for Task 2 (the table input and the table-scope revoke are already in the cell):

```python
try:
    glue.create_table(
        DatabaseName=DATABASE_NAME,
        TableInput=table_input,
    )
    print(f"Glue table created: {DATABASE_NAME}.{TABLE_NAME}")
except ClientError as e:
    if e.response["Error"]["Code"] == "AlreadyExistsException":
        print(f"Glue table {DATABASE_NAME}.{TABLE_NAME} already exists, reusing.")
    else:
        raise
```

If `create_table` returns `AccessDeniedException` and the caller is the lab user, Lake Formation governance is now blocking direct catalog writes. Confirm the setup cell ran `put_data_lake_settings` with the caller as a data lake admin; without admin status the caller cannot create tables in a database under LF governance. The setup cell is idempotent, so re-running it is the fix.

</details>


**Wallet check.** Still at $0.00. Creating a Glue table is a control-plane call against the Data Catalog, which is free at this scale (the first 1 million catalog objects and 1 million requests per month are always-free). The Lake Formation revoke is also a control-plane call and accrues no charge.


## Task 3: Create the analytics IAM role and grant column-level `SELECT` on `customers` excluding `ssn` and `email`

This is the heart of the lab. Once the analytics role exists and the grant is in place, the next task proves the boundary by assuming the role and running Athena queries.

Three pieces of plumbing to land in this task:

1. **Create the analytics IAM role** with a trust policy that allows the account root to assume it. Account-root trust is the simplest pattern: the labrun-test IAM user (and any other principal in the account) gets `sts:AssumeRole` via its own attached policy, and the role itself trusts the account. Tag the role with the lab tag.
2. **Attach the Athena and S3 read managed policies** to the analytics role. The role needs `AmazonAthenaFullAccess` to call Athena and `AmazonS3ReadOnlyAccess` plus an inline grant for Athena query result writes. (Athena needs to write its own results back to S3 even when the underlying data is governed by LF.)
3. **Grant the analytics role table-with-columns SELECT** via `lakeformation.grant_permissions`. The `Resource.TableWithColumns` form carries `ColumnWildcard.ExcludedColumnNames=["ssn", "email"]`, which expresses "everything except the two PII columns." The exact grant body is also stored in `LF_GRANT_BODIES` so the cleanup adapter can issue the matching `revoke_permissions` call at teardown.

Lake Formation also accepts the explicit `ColumnNames=["customer_id", "signup_date", "country"]` form, which means the same thing. Either form satisfies Checkpoint 3; the wildcard pattern is generally cleaner because adding a new non-PII column does not require updating the grant.


In [ ]:
# NBVAL_SKIP
# Task 3: Create the analytics IAM role with account-root trust, attach the
# managed Athena and S3 read policies plus an Athena results inline grant,
# then grant the analytics role table-with-columns SELECT on customers with
# ssn and email excluded.

iam = boto3.client(
    "iam",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)
athena = boto3.client(
    "athena",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)
lakeformation = boto3.client(
    "lakeformation",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)

# Trust policy: account root can assume this role. The labrun-test IAM user
# attached to the lab session inherits sts:AssumeRole via IAMFullAccess (or
# the labrun-sts-assume inline grant), so the lab cell can assume the role
# without a per-user Principal entry.
account_root_arn = f"arn:aws:iam::{ACCOUNT_ID}:root"
trust_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Principal": {"AWS": account_root_arn},
            "Action": "sts:AssumeRole",
        }
    ],
}

# YOUR CODE: Create the analytics role with iam.create_role(
#     RoleName=ANALYTICS_ROLE_NAME,
#     AssumeRolePolicyDocument=json.dumps(trust_policy),
#     Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
# ).
# Wrap in try/except ClientError; on EntityAlreadyExists update the trust
# policy with iam.update_assume_role_policy.

print(f"Analytics role created: {ANALYTICS_ROLE_NAME}")

# Attach AWS-managed policies the role needs at query time. AmazonAthenaFullAccess
# lets it call Athena; AmazonS3ReadOnlyAccess lets it read the Parquet at S3
# scan time (LF still gates the column projection). Athena also needs write
# access to its results location, granted by the inline policy below.
iam.attach_role_policy(
    RoleName=ANALYTICS_ROLE_NAME,
    PolicyArn="arn:aws:iam::aws:policy/AmazonAthenaFullAccess",
)
iam.attach_role_policy(
    RoleName=ANALYTICS_ROLE_NAME,
    PolicyArn="arn:aws:iam::aws:policy/AmazonS3ReadOnlyAccess",
)

# Inline policy: write Athena results to the lab bucket. Athena always writes
# query results back to S3 even when the underlying data is LF-governed.
athena_results_prefix_arn = f"arn:aws:s3:::{BUCKET_NAME}/athena-results/*"
bucket_arn = f"arn:aws:s3:::{BUCKET_NAME}"
athena_inline = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Action": [
                "s3:PutObject",
                "s3:GetObject",
                "s3:GetBucketLocation",
                "s3:ListBucket",
                "s3:ListBucketMultipartUploads",
                "s3:AbortMultipartUpload",
                "s3:ListMultipartUploadParts",
            ],
            "Resource": [bucket_arn, athena_results_prefix_arn],
        }
    ],
}
iam.put_role_policy(
    RoleName=ANALYTICS_ROLE_NAME,
    PolicyName=f"labrun-{LAB_ID}-athena-results",
    PolicyDocument=json.dumps(athena_inline),
)
print("Analytics role policies attached (AmazonAthenaFullAccess, AmazonS3ReadOnlyAccess, athena-results inline).")

# Create the Athena workgroup. The workgroup writes results to s3://{BUCKET}/athena-results/
# and the EnforceWorkGroupConfiguration flag forces every query (including
# the assumed-role queries in Task 4) to use this OutputLocation.
athena_results_uri = f"s3://{BUCKET_NAME}/athena-results/"
try:
    athena.create_work_group(
        Name=WORKGROUP_NAME,
        Configuration={
            "ResultConfiguration": {"OutputLocation": athena_results_uri},
            "EnforceWorkGroupConfiguration": True,
            "PublishCloudWatchMetricsEnabled": False,
        },
        Description=f"Lab workgroup for {LAB_ID}.",
        Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
    )
    print(f"Athena workgroup created: {WORKGROUP_NAME}")
except ClientError as e:
    code = e.response["Error"]["Code"]
    if code == "InvalidRequestException":
        athena.update_work_group(
            WorkGroup=WORKGROUP_NAME,
            ConfigurationUpdates={
                "ResultConfigurationUpdates": {"OutputLocation": athena_results_uri},
                "EnforceWorkGroupConfiguration": True,
            },
            State="ENABLED",
        )
        print(f"Athena workgroup {WORKGROUP_NAME} already exists, reused with updated config.")
    else:
        raise

# IAM role propagation. LF grant + STS assume in Task 4 both validate the
# role; skipping this sleep produces flaky InvalidParameterValueException
# failures on assume_role and confusing GrantPermissions errors on the LF
# call. Five seconds is plenty for the role to settle.
print("Letting the IAM role propagate for 10 seconds...")
time.sleep(10)
print("Role is ready.")

analytics_role_arn = f"arn:aws:iam::{ACCOUNT_ID}:role/{ANALYTICS_ROLE_NAME}"

# Build the grant body once so the cleanup adapter can issue the exact
# revoke_permissions call at teardown. LF requires the precise grant body
# to revoke; a stringified CLI command in cli_delete_command is for
# documentation only.
grant_body = {
    "Principal": {"DataLakePrincipalIdentifier": analytics_role_arn},
    "Resource": {
        "TableWithColumns": {
            "DatabaseName": DATABASE_NAME,
            "Name": TABLE_NAME,
            "ColumnWildcard": {"ExcludedColumnNames": PII_COLUMNS},
        }
    },
    "Permissions": ["SELECT"],
    "PermissionsWithGrantOption": [],
}

grant_key = f"analytics-role:{DATABASE_NAME}.{TABLE_NAME}"
LF_GRANT_BODIES[grant_key] = grant_body

# YOUR CODE: Call lakeformation.grant_permissions(**grant_body) to grant the
# analytics role table-with-columns SELECT with ssn and email excluded.
# Wrap in try/except ClientError; on InvalidInputException ("already
# granted") swallow the error so a re-run does not abort.

print(f"Lake Formation SELECT grant issued to {analytics_role_arn}")
print(f"  Resource: TableWithColumns {DATABASE_NAME}.{TABLE_NAME}")
print(f"  Excluded columns: {PII_COLUMNS}")


In [ ]:
# NBVAL_SKIP
# Checkpoint 3: analytics role holds a Lake Formation SELECT grant on the
# customers table with ssn and email excluded. The validator accepts either
# the ColumnWildcard.ExcludedColumnNames form (recommended) or the explicit
# ColumnNames form listing the three non-PII columns.


def checkpoint_3(session):
    try:
        lf = boto3.client(
            "lakeformation",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )

        analytics_role_arn = f"arn:aws:iam::{ACCOUNT_ID}:role/{ANALYTICS_ROLE_NAME}"
        try:
            resp = lf.list_permissions(
                Principal={"DataLakePrincipalIdentifier": analytics_role_arn},
                Resource={
                    "TableWithColumns": {
                        "DatabaseName": DATABASE_NAME,
                        "Name": TABLE_NAME,
                        "ColumnWildcard": {"ExcludedColumnNames": PII_COLUMNS},
                    }
                },
            )
        except ClientError as e:
            return CheckpointResult(
                status="error",
                error_reason=f"list_permissions failed for analytics role: {e}",
            )

        grants = resp.get("PrincipalResourcePermissions", [])
        if not grants:
            # Try the explicit-ColumnNames form (the other accepted shape).
            try:
                alt = lf.list_permissions(
                    Principal={"DataLakePrincipalIdentifier": analytics_role_arn},
                    Resource={
                        "Table": {"DatabaseName": DATABASE_NAME, "Name": TABLE_NAME},
                    },
                )
                grants = alt.get("PrincipalResourcePermissions", [])
            except ClientError:
                grants = []

        if not grants:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"No Lake Formation grant found for {analytics_role_arn} "
                    f"on {DATABASE_NAME}.{TABLE_NAME}. Run "
                    f"lakeformation.grant_permissions with the grant_body "
                    f"dict from the Task 3 cell."
                ),
            )

        # Find a SELECT grant on the table with PII excluded.
        for grant in grants:
            perms = set(grant.get("Permissions", []))
            if "SELECT" not in perms:
                continue
            res = grant.get("Resource", {})
            twc = res.get("TableWithColumns", {})
            if not twc:
                # Some grants come back under Resource.Table; treat as table-level
                # SELECT, which is too broad (PII would be readable). Fail.
                continue
            if twc.get("DatabaseName") != DATABASE_NAME or twc.get("Name") != TABLE_NAME:
                continue
            wildcard = twc.get("ColumnWildcard")
            column_names = twc.get("ColumnNames")
            if wildcard:
                excluded = set(wildcard.get("ExcludedColumnNames", []))
                if {"ssn", "email"}.issubset(excluded):
                    return CheckpointResult(status="pass")
            elif column_names:
                col_set = set(column_names)
                if "ssn" not in col_set and "email" not in col_set and set(NON_PII_COLUMNS).issubset(col_set):
                    return CheckpointResult(status="pass")

        return CheckpointResult(
            status="fail",
            error_reason=(
                f"Found grant(s) for {analytics_role_arn} on {DATABASE_NAME}.{TABLE_NAME} "
                f"but none match the required shape: SELECT on TableWithColumns "
                f"with ssn and email excluded (or with ColumnNames listing only "
                f"the three non-PII columns). Inspect list_permissions output."
            ),
        )
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(3, checkpoint_3)


<details><summary>Hint 1 (nudge)</summary>

The checkpoint walks the analytics role's grants and looks for one with `Permissions=["SELECT"]` on `Resource.TableWithColumns` for the customers table where `ColumnWildcard.ExcludedColumnNames` contains both `ssn` and `email`. The explicit `ColumnNames` form listing only the three non-PII columns also passes. The most common miss is granting `Resource.Table` (table-level SELECT) instead of `Resource.TableWithColumns`; a plain table grant is too broad because it includes the PII columns and the validator rejects it.

</details>

<details><summary>Hint 2 (direction)</summary>

Two API calls hold the win. `iam.create_role` builds the analytics role with the prebuilt `trust_policy`; the rest of the cell handles managed and inline policy attaches plus the workgroup. `lakeformation.grant_permissions(**grant_body)` issues the column-level SELECT. The `grant_body` dict is already constructed for you with `ColumnWildcard.ExcludedColumnNames=PII_COLUMNS`. Wrap the LF call in try/except `ClientError` and on `InvalidInputException` swallow the "already granted" error.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Here is the complete working solution for Task 3 (the trust policy, the managed policy attaches, the inline policy attach, the workgroup, and the `grant_body` dict are already in the cell):

```python
try:
    iam.create_role(
        RoleName=ANALYTICS_ROLE_NAME,
        AssumeRolePolicyDocument=json.dumps(trust_policy),
        Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
    )
except ClientError as e:
    if e.response["Error"]["Code"] == "EntityAlreadyExists":
        iam.update_assume_role_policy(
            RoleName=ANALYTICS_ROLE_NAME,
            PolicyDocument=json.dumps(trust_policy),
        )
    else:
        raise

try:
    lakeformation.grant_permissions(**grant_body)
except ClientError as e:
    if e.response["Error"]["Code"] != "InvalidInputException":
        raise
```

If `grant_permissions` returns `AccessDeniedException`, the caller is not a Lake Formation data lake admin. Re-run the setup cell; `put_data_lake_settings` is idempotent and fixes admin status. If the explicit `ColumnNames=NON_PII_COLUMNS` form is preferred, swap `ColumnWildcard` for `ColumnNames` inside `grant_body.Resource.TableWithColumns`; either form passes Checkpoint 3.

</details>


**Wallet check.** Still effectively $0.00. IAM role and policy operations are free at any volume. Lake Formation grants and the Athena workgroup are control-plane calls with no per-action charge. The role propagation sleep is wall-clock only, not billable.


## Task 4: Assume the analytics role and prove PII reads are blocked while non-PII reads succeed

The grant is in place. Now prove the boundary by assuming the analytics role, building a fresh Athena client from the temporary credentials, and running two queries.

Three pieces of plumbing to land in this task:

1. **Assume the analytics role** via `sts.assume_role`. The trust policy in Task 3 allows account-root, so the labrun-test IAM user (which already has `sts:AssumeRole` via `IAMFullAccess` or the `labrun-sts-assume` inline grant) can make the call without further setup.
2. **Build a fresh Athena client** from the temporary credentials returned by STS. The new client must carry `aws_access_key_id`, `aws_secret_access_key`, AND `aws_session_token` sourced from `temp_creds["AccessKeyId"]`, `temp_creds["SecretAccessKey"]`, `temp_creds["SessionToken"]`. A client built without the temp session token tests the lab caller's permissions, not the analytics role's.
3. **Run two queries through the assumed-role Athena client**:
   - `SELECT customer_id, country FROM customers LIMIT 10` should succeed (LF grants SELECT on these columns).
   - `SELECT ssn FROM customers LIMIT 10` should fail with `QueryExecution.Status.State == "FAILED"` and a `StateChangeReason` that contains one of `"AccessDenied"`, `"Insufficient permissions"`, or `"HIVE_PARTITION_SCHEMA_MISMATCH"` (Lake Formation surfaces the denial through different Athena error categories depending on engine version; all three forms are LF denials in disguise).

The pass-then-fail pattern is the whole point: column-level grants let you publish the same table to multiple roles with different visibility, no view-shimming, no copies, no consumer-side filtering.


In [ ]:
# NBVAL_SKIP
# Task 4: Assume the analytics role, build a fresh Athena client from the
# temp credentials, and run the two comparison queries. Stash the resulting
# state objects so the checkpoint cell can read them without re-running.

sts_client = boto3.client(
    "sts",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)

analytics_role_arn = f"arn:aws:iam::{ACCOUNT_ID}:role/{ANALYTICS_ROLE_NAME}"

# YOUR CODE: Call sts_client.assume_role() with RoleArn=analytics_role_arn and
# RoleSessionName="labrun-column-permissions-test". Save the response to
# assume_resp.

temp_creds = assume_resp["Credentials"]
print(f"Assumed role: {analytics_role_arn}")
print(f"Temporary credentials expire at: {temp_creds['Expiration'].isoformat()}")

# YOUR CODE: Build a new Athena client from the assumed-role temporary
# credentials. Pass aws_access_key_id, aws_secret_access_key, AND
# aws_session_token sourced from temp_creds (keys: AccessKeyId,
# SecretAccessKey, SessionToken) plus region_name=REGION. Save to
# analytics_athena.


def run_query_as_analytics(sql: str, athena_client) -> dict:
    """Submit one query as the analytics role and poll until terminal.

    Returns the full QueryExecution dict. Does NOT raise on FAILED; the
    caller inspects Status.State to interpret the result. This is the
    deny-test pattern: a FAILED with the right StateChangeReason is the
    expected outcome for the PII query.
    """
    start = athena_client.start_query_execution(
        QueryString=sql,
        QueryExecutionContext={"Database": DATABASE_NAME},
        WorkGroup=WORKGROUP_NAME,
    )
    qid = start["QueryExecutionId"]
    deadline = time.time() + 120
    while time.time() < deadline:
        resp = athena_client.get_query_execution(QueryExecutionId=qid)
        state = resp["QueryExecution"]["Status"]["State"]
        if state in ("SUCCEEDED", "FAILED", "CANCELLED"):
            return resp["QueryExecution"]
        time.sleep(2)
    raise RuntimeError(f"Query {qid} did not finish within 2 minutes.")


# Half 1: non-PII columns. This query should succeed and return rows.
NON_PII_SQL = f"SELECT customer_id, country FROM {TABLE_NAME} LIMIT 10"
print(f"Running non-PII query as analytics role: {NON_PII_SQL}")
non_pii_exec = run_query_as_analytics(NON_PII_SQL, analytics_athena)
NON_PII_STATE = non_pii_exec["Status"]["State"]
NON_PII_REASON = non_pii_exec["Status"].get("StateChangeReason", "")
print(f"  State: {NON_PII_STATE}")
if NON_PII_REASON:
    print(f"  StateChangeReason: {NON_PII_REASON}")

# Half 2: PII columns. This query should FAIL with an LF denial.
PII_SQL = f"SELECT ssn FROM {TABLE_NAME} LIMIT 10"
print(f"Running PII query as analytics role: {PII_SQL}")
pii_exec = run_query_as_analytics(PII_SQL, analytics_athena)
PII_STATE = pii_exec["Status"]["State"]
PII_REASON = pii_exec["Status"].get("StateChangeReason", "")
print(f"  State: {PII_STATE}")
if PII_REASON:
    print(f"  StateChangeReason: {PII_REASON}")

# Interpret in plain English.
print()
if NON_PII_STATE == "SUCCEEDED" and PII_STATE == "FAILED":
    print("PASS-shape: non-PII query succeeded, PII query failed. Column-level grant boundary holds.")
else:
    print("Result is not the expected pass-then-fail shape. Re-check the LF grant in Task 3.")


In [ ]:
# NBVAL_SKIP
# Checkpoint 4: assume the analytics role, build a fresh Athena client, and
# verify the two-query deny-test independently of cell ordering. Non-PII
# query must SUCCEED; PII query must FAIL with one of the recognized LF
# denial substrings in StateChangeReason.

LF_DENIAL_SUBSTRINGS = (
    "AccessDenied",
    "Insufficient permissions",
    "HIVE_PARTITION_SCHEMA_MISMATCH",
)


def checkpoint_4(session):
    try:
        sts_check = boto3.client(
            "sts",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )
        role_arn = f"arn:aws:iam::{ACCOUNT_ID}:role/{ANALYTICS_ROLE_NAME}"
        try:
            assume_resp_cp = sts_check.assume_role(
                RoleArn=role_arn,
                RoleSessionName="labrun-checkpoint-4",
            )
        except ClientError as e:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Could not assume {role_arn}. STS returned "
                    f"{e.response['Error']['Code']}: {e.response['Error']['Message']}. "
                    f"Confirm the trust policy Principal includes "
                    f"arn:aws:iam::{ACCOUNT_ID}:root."
                ),
            )

        temp = assume_resp_cp["Credentials"]
        athena_cp = boto3.client(
            "athena",
            aws_access_key_id=temp["AccessKeyId"],
            aws_secret_access_key=temp["SecretAccessKey"],
            aws_session_token=temp["SessionToken"],
            region_name=REGION,
        )

        def _run(sql: str) -> dict:
            start = athena_cp.start_query_execution(
                QueryString=sql,
                QueryExecutionContext={"Database": DATABASE_NAME},
                WorkGroup=WORKGROUP_NAME,
            )
            qid = start["QueryExecutionId"]
            deadline = time.time() + 120
            while time.time() < deadline:
                resp = athena_cp.get_query_execution(QueryExecutionId=qid)
                state = resp["QueryExecution"]["Status"]["State"]
                if state in ("SUCCEEDED", "FAILED", "CANCELLED"):
                    return resp["QueryExecution"]
                time.sleep(2)
            raise RuntimeError(f"Query {qid} did not finish within 2 minutes.")

        # Non-PII query: must SUCCEED.
        non_pii_sql = f"SELECT customer_id, country FROM {TABLE_NAME} LIMIT 10"
        try:
            non_pii = _run(non_pii_sql)
        except ClientError as e:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Non-PII query could not start: {e}. Confirm the analytics "
                    f"role has AmazonAthenaFullAccess attached and the workgroup "
                    f"{WORKGROUP_NAME!r} exists."
                ),
            )
        non_pii_state = non_pii["Status"]["State"]
        if non_pii_state != "SUCCEEDED":
            reason = non_pii["Status"].get("StateChangeReason", "(no reason)")
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Non-PII query ended {non_pii_state}: {reason}. The analytics "
                    f"role should be able to read customer_id and country. Re-check "
                    f"the LF grant in Task 3 and confirm the IAMAllowedPrincipals "
                    f"revoke from Task 2 fired."
                ),
            )

        # PII query: must FAIL with an LF denial substring.
        pii_sql = f"SELECT ssn FROM {TABLE_NAME} LIMIT 10"
        try:
            pii = _run(pii_sql)
        except ClientError as e:
            return CheckpointResult(
                status="fail",
                error_reason=f"PII query could not start: {e}.",
            )
        pii_state = pii["Status"]["State"]
        pii_reason = pii["Status"].get("StateChangeReason", "")
        if pii_state != "FAILED":
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"PII query ended {pii_state} (reason: {pii_reason!r}), "
                    f"expected FAILED. The analytics role should NOT be able to "
                    f"read ssn. The LF grant likely uses Resource.Table (table-level "
                    f"SELECT) instead of Resource.TableWithColumns; re-check Task 3."
                ),
            )

        if not any(s in pii_reason for s in LF_DENIAL_SUBSTRINGS):
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"PII query failed but StateChangeReason is {pii_reason!r}; "
                    f"expected to contain one of {LF_DENIAL_SUBSTRINGS}. The query "
                    f"failed for a different reason; investigate before declaring "
                    f"the boundary intact."
                ),
            )

        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(4, checkpoint_4)


<details><summary>Hint 1 (nudge)</summary>

The checkpoint runs the two queries independently of your Task 4 cell, so a missed cell run is not the failure mode. Read the failure reason. The most common miss on the first run is forgetting `aws_session_token` on the assumed-role Athena client; without it the client falls back to the lab caller's credentials and the PII query succeeds (because the lab caller has full S3 read on the bucket). The second most common miss is granting `Resource.Table` instead of `Resource.TableWithColumns` in Task 3; table-level SELECT is too broad and the PII query also succeeds.

</details>

<details><summary>Hint 2 (direction)</summary>

Two API calls drive the test. `sts_client.assume_role(RoleArn=analytics_role_arn, RoleSessionName="labrun-column-permissions-test")` returns temporary credentials in `assume_resp["Credentials"]` with `AccessKeyId`, `SecretAccessKey`, `SessionToken`. Build the new Athena client with `boto3.client("athena", aws_access_key_id=temp_creds["AccessKeyId"], aws_secret_access_key=temp_creds["SecretAccessKey"], aws_session_token=temp_creds["SessionToken"], region_name=REGION)`. The two queries run through the helper `run_query_as_analytics` already in the cell.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Here is the complete working solution for Task 4 (the helper function and the two query strings are already in the cell):

```python
assume_resp = sts_client.assume_role(
    RoleArn=analytics_role_arn,
    RoleSessionName="labrun-column-permissions-test",
)
temp_creds = assume_resp["Credentials"]

analytics_athena = boto3.client(
    "athena",
    aws_access_key_id=temp_creds["AccessKeyId"],
    aws_secret_access_key=temp_creds["SecretAccessKey"],
    aws_session_token=temp_creds["SessionToken"],
    region_name=REGION,
)
```

If `assume_role` returns `AccessDenied` with "is not authorized to perform: sts:AssumeRole," the lab caller's attached policies do not allow assume against the analytics role ARN. `IAMFullAccess` covers it; the cross-lab inline `labrun-sts-assume` also covers it. If the non-PII query is still failing under the assumed role with "Database does not exist," the Athena client is pointing at the default database; pass `QueryExecutionContext={"Database": DATABASE_NAME}` to `start_query_execution`.

</details>


**Wallet check.** Still under one cent total for the session. STS assume-role calls are free. Each Athena query against the 100-row Parquet scans well under one kilobyte, which is essentially free against the $5-per-terabyte Athena rate. The morning coffee continues to win.


In [ ]:
# NBVAL_SKIP
# Cleanup. Tear down every resource in CLEANUP_MANIFEST in reverse-creation
# order. Lab 11 has no critical (hourly-billed) resources. Lake Formation
# grants and registrations accrue no per-hour charge but are account-level
# state that MUST be explicitly cleaned up; dangling grants are confusing
# on the next audit.
#
# labrun-checks 0.3.0 ships AWS adapters for s3_bucket, iam_role,
# glue_database, and glue_table. It does NOT yet support
# lakeformation_permissions, lakeformation_resource, or athena_workgroup.
# A _LabAdapter subclass wraps AwsCleanupAdapter and adds those three
# resource types inline. When the package promotes them in a future
# release, the wrapper can be removed and run_cleanup called against the
# default.
#
# The lakeformation_permissions adapter is the trickiest: LF requires the
# exact structured grant body to revoke (Principal, Resource, Permissions).
# Task 3 stored that body in LF_GRANT_BODIES keyed by CleanupResource.id,
# and the adapter looks it up at teardown.

import sys
from labrun_checks.adapters.aws import AwsCleanupAdapter


class _LabAdapter:
    """AwsCleanupAdapter wrapper that adds lakeformation_permissions,
    lakeformation_resource, and athena_workgroup support."""

    def __init__(self) -> None:
        self._aws = AwsCleanupAdapter()

    def _lf_client(self, credentials: dict):
        return boto3.client(
            "lakeformation",
            aws_access_key_id=credentials["aws_access_key_id"],
            aws_secret_access_key=credentials["aws_secret_access_key"],
            aws_session_token=credentials.get("aws_session_token"),
            region_name=credentials.get("region", REGION),
        )

    def _athena_client(self, credentials: dict):
        return boto3.client(
            "athena",
            aws_access_key_id=credentials["aws_access_key_id"],
            aws_secret_access_key=credentials["aws_secret_access_key"],
            aws_session_token=credentials.get("aws_session_token"),
            region_name=credentials.get("region", REGION),
        )

    def delete_resource(self, credentials: dict, resource) -> None:
        if resource.type == "lakeformation_permissions":
            client = self._lf_client(credentials)
            body = LF_GRANT_BODIES.get(resource.id)
            if not body:
                # Nothing stashed for this id. Treat as already-revoked rather
                # than raising; the manifest entry covers a grant that was
                # never issued in this session.
                return
            try:
                client.revoke_permissions(**body)
            except ClientError as e:
                code = e.response["Error"]["Code"]
                if code in ("InvalidInputException", "EntityNotFoundException"):
                    return
                raise
            return
        if resource.type == "lakeformation_resource":
            client = self._lf_client(credentials)
            try:
                client.deregister_resource(ResourceArn=resource.id)
            except ClientError as e:
                code = e.response["Error"]["Code"]
                if code in ("EntityNotFoundException", "InvalidInputException"):
                    return
                raise
            return
        if resource.type == "athena_workgroup":
            client = self._athena_client(credentials)
            try:
                client.delete_work_group(
                    WorkGroup=resource.id,
                    RecursiveDeleteOption=True,
                )
            except ClientError as e:
                code = e.response["Error"]["Code"]
                if code in ("InvalidRequestException", "ResourceNotFoundException"):
                    return
                raise
            return
        return self._aws.delete_resource(credentials, resource)

    def describe_resource(self, credentials: dict, resource) -> bool:
        if resource.type == "lakeformation_permissions":
            client = self._lf_client(credentials)
            body = LF_GRANT_BODIES.get(resource.id)
            if not body:
                return False
            try:
                resp = client.list_permissions(
                    Principal=body["Principal"],
                    Resource=body["Resource"],
                )
            except ClientError as e:
                code = e.response["Error"]["Code"]
                if code in ("EntityNotFoundException", "InvalidInputException"):
                    return False
                raise
            return bool(resp.get("PrincipalResourcePermissions"))
        if resource.type == "lakeformation_resource":
            client = self._lf_client(credentials)
            paginator = client.get_paginator("list_resources")
            for page in paginator.paginate():
                for r in page.get("ResourceInfoList", []):
                    if r.get("ResourceArn", "").rstrip("/") == resource.id.rstrip("/"):
                        return True
            return False
        if resource.type == "athena_workgroup":
            client = self._athena_client(credentials)
            try:
                client.get_work_group(WorkGroup=resource.id)
                return True
            except ClientError as e:
                code = e.response["Error"]["Code"]
                if code in ("InvalidRequestException", "ResourceNotFoundException"):
                    return False
                raise
        return self._aws.describe_resource(credentials, resource)

    def tag_scan(self, credentials: dict, lab_slug: str, region: str) -> list[str]:
        return self._aws.tag_scan(credentials, lab_slug, region)


# Empty the S3 bucket before run_cleanup tears it down. S3 buckets cannot be
# deleted while they contain objects; the bundled s3_bucket adapter only
# deletes one page of objects at a time, and the lab wrote the customers
# Parquet plus the athena-results prefix plus query metadata.
print("Emptying the S3 bucket before teardown...")
s3 = boto3.client(
    "s3",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)
try:
    paginator = s3.get_paginator("list_objects_v2")
    for page in paginator.paginate(Bucket=BUCKET_NAME):
        keys = [{"Key": obj["Key"]} for obj in page.get("Contents", [])]
        if keys:
            s3.delete_objects(Bucket=BUCKET_NAME, Delete={"Objects": keys})
except ClientError as e:
    if e.response["Error"]["Code"] != "NoSuchBucket":
        print(f"Bucket emptying ran into: {e}. Continuing to cleanup.")

result = run_cleanup(CLEANUP_MANIFEST, adapter=_LabAdapter())

for warning in result.warnings:
    print(warning)
for orphan in result.orphans:
    print(orphan)
if result.warnings or result.orphans:
    print()

failed_ids: set[str] = set()
for w in result.warnings:
    for r in CLEANUP_MANIFEST:
        if r.id in w:
            failed_ids.add(r.id)
            break

critical_total = 0
standard_total = len(CLEANUP_MANIFEST)
critical_destroyed = critical_total
standard_destroyed = standard_total - len(failed_ids)
failed_count = len(failed_ids)

print("Cleanup complete.")
print(f"Critical resources destroyed: {critical_destroyed}")
print(f"Standard resources destroyed: {standard_destroyed}")
print(f"Resources that failed to delete: {failed_count} (see above for CLI commands)")
print("If K > 0, your AWS account may still incur charges. Resolve before closing this notebook.")

cleanup(status=result.status)

if failed_count > 0:
    sys.exit(1)


**Session total: under $0.05.** Lake Formation does not charge for grants or registrations; you paid for the underlying S3 storage and requests (kilobytes), the Glue Data Catalog (free at this scale), and three or four small Athena queries that scanned well under a kilobyte each. Everything else (IAM, STS, control-plane calls) is free. The cleanup cell above revoked the LF grant, deregistered the S3 location, deleted the workgroup, dropped the Glue table and database, deleted the analytics role, and emptied and deleted the bucket so no dangling grants remain on the next audit.


## These are not graded. They are for you.

Three questions worth sitting with before the exam.

1. You granted column-level SELECT with `ColumnWildcard.ExcludedColumnNames=["ssn", "email"]` in this lab. The explicit `ColumnNames=["customer_id", "signup_date", "country"]` form is equivalent today. Walk through what happens to each form when a new column `phone_number` is added to the customers table next week. Which form leaks PII by default? Which form silently drops the new column from the analytics view? This is the kind of "grant maintenance" trade-off DEA-C01 loves to test.

2. The `IAMAllowedPrincipals` default grant is the single biggest source of confusion in Lake Formation. AWS adds it to every new database and table to keep legacy IAM-based access working during migration. Walk through what happens if you grant column-level SELECT to the analytics role but forget to revoke `IAMAllowedPrincipals` from the customers table. Hint: the query against `ssn` succeeds, the audit catches nothing, and the column-level grant feels like it does not work. This question maps directly to DEA-C01 Domain 4 governance scenarios.

3. The analytics role's trust policy in this lab uses account-root (`arn:aws:iam::{account}:root`). Compare this to a tighter trust policy that names the labrun-test IAM user explicitly. When is account-root acceptable; when is it a security smell; what is the practical difference for an attacker who already has account-level credentials? This is the same trade-off you will see in cross-account-trust scenario questions on the exam.

## Exam-Style Practice

**Q1.** A data engineer grants the analytics IAM role `Resource.Table` (table-level SELECT) in Lake Formation on the customers table, intending the role to be unable to read `ssn` and `email`. Athena queries from the analytics role return `ssn` values successfully. What is the most likely cause?

A) Lake Formation table-level SELECT grants the entire table including all columns; the engineer needed `Resource.TableWithColumns` with `ExcludedColumnNames=["ssn", "email"]` for column-level exclusion.

B) Athena bypasses Lake Formation column grants for `string`-typed columns; switch the `ssn` column to `varchar`.

C) The Glue Data Catalog caches column projections for 24 hours; wait a day and re-run the query.

D) The labrun-test IAM user has `AmazonS3FullAccess`, which overrides Lake Formation grants on the underlying bucket.

<details><summary>Show answer</summary>

**A**.

- A) Right. `Resource.Table` grants SELECT on every column in the table. To exclude specific columns, the grant must use `Resource.TableWithColumns` with `ColumnWildcard.ExcludedColumnNames` (or the equivalent explicit `ColumnNames` listing only the allowed columns). This is the most common LF authoring mistake and shows up directly on DEA-C01 governance questions.
- B) Wrong. Column type is irrelevant to LF column grants. LF keys on column names, not types.
- C) Wrong. Glue Data Catalog and Athena do cache plan-time metadata briefly, but not column-level grant decisions. A re-run does not change the LF authorization result.
- D) Wrong. Once an S3 location is registered with Lake Formation, IAM policies on the bucket no longer apply at LF-governed query time; LF mediates access. `AmazonS3FullAccess` would matter if the analytics role queried S3 directly, but Athena routes through LF here.

</details>

**Q2.** A new analytics team table `events` is added to a database that is already under Lake Formation governance. The data engineer issues `lakeformation.grant_permissions` to the analytics role with column-level SELECT excluding `user_email`. Queries from the analytics role still return `user_email` values. The `IAMAllowedPrincipals` grant was already revoked on the database before the table was added. What is the most likely cause?

A) Newly created tables in an LF-governed database inherit an `IAMAllowedPrincipals` default grant at the table scope even if the database-scope default was revoked. The fix is to revoke `IAMAllowedPrincipals` on the new `events` table specifically.

B) The Glue Data Catalog automatically promotes column-level grants to table-level grants after 24 hours.

C) `lakeformation.grant_permissions` requires `iam:PassRole` on the analytics role and the call silently fails without it.

D) Athena's engine version 3 ignores `ColumnWildcard.ExcludedColumnNames` and only honors explicit `ColumnNames` lists; switch the grant form.

<details><summary>Show answer</summary>

**A**.

- A) Right. AWS attaches an `IAMAllowedPrincipals` default grant at the database scope AND at the table scope when each is created. Revoking the database-scope default does not retroactively revoke the table-scope defaults on tables that already existed, and it does not prevent the table-scope default from being attached when a new table is created. Every new table needs its own `IAMAllowedPrincipals` revoke right after creation.
- B) Wrong. There is no such automatic promotion. Column-level grants stay column-level.
- C) Wrong. `iam:PassRole` is required for service-linked roles in `register_resource`, not for `grant_permissions`. A silent failure on grant_permissions does not happen; the call would raise.
- D) Wrong. Both engine versions honor both `ColumnWildcard.ExcludedColumnNames` and explicit `ColumnNames`. The two forms are equivalent and both pass the validator in this lab.

</details>

**Q3.** An analytics IAM role is granted Lake Formation column-level SELECT on a Parquet-backed table with `ssn` excluded. The role can read the non-PII columns from Athena. The same role tries to read the table directly with `boto3.client("s3").get_object(Bucket=B, Key="customers/customers.parquet")` to bypass Athena and is successful. What change closes the bypass?

A) Add a Lake Formation `LF-Tag` on the `ssn` column so S3 reads are blocked at the object level.

B) Register the S3 location with Lake Formation using `UseServiceLinkedRole=True` and revoke `s3:GetObject` on the underlying bucket from any principal except the LF service-linked role.

C) Switch the Parquet file format to ORC; ORC natively enforces LF column grants on read.

D) Replace the analytics role's `AmazonS3ReadOnlyAccess` with an inline policy that denies `s3:GetObject` on `customers/*`.

<details><summary>Show answer</summary>

**B**.

- A) Wrong. LF-Tags govern catalog-level access decisions, not S3 object-level reads. LF cannot enforce column-level filtering on a raw S3 GetObject; it can only block the GetObject entirely.
- B) Right. Lake Formation column-level grants only apply when the query routes through an LF-aware engine like Athena, Redshift Spectrum, EMR with LF, or Glue ETL with LF. A raw `s3:GetObject` against the underlying object bypasses LF entirely. The fix is to register the S3 location with LF and remove all direct IAM-based S3 read paths to the underlying objects, so only the LF service-linked role can read S3 and only LF-aware query engines can reach the data. This is a recurring "LF is not enough on its own" theme on DEA-C01.
- C) Wrong. ORC and Parquet have the same LF enforcement story. Neither file format enforces grants at read time; the query engine plus LF does.
- D) Wrong. A per-role deny on `customers/*` closes this bypass for the analytics role specifically but leaves the door open for any other role that picks up `AmazonS3ReadOnlyAccess` later. The architectural fix is the LF registration plus removing direct IAM S3 reads from the bucket entirely.

</details>
